In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [7]:
# Load & quick inspect
data = pd.read_csv(r"D:\tableau\Assignment\Group Assignment\Original data.csv")
print('Data read into a pandas dataframe!')

Data read into a pandas dataframe!


In [9]:
# Display features
data.head()

# Check for missing values
data.info()

# Summary
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Country                         500 non-null    object 
 1   Year                            500 non-null    int64  
 2   Average_Monthly_Income          500 non-null    float64
 3   Cost_of_Living                  500 non-null    float64
 4   Housing_Cost_Percentage         500 non-null    float64
 5   Tax_Rate                        500 non-null    float64
 6   Savings_Percentage              500 non-null    float64
 7   Healthcare_Cost_Percentage      500 non-null    float64
 8   Education_Cost_Percentage       500 non-null    float64
 9   Transportation_Cost_Percentage  500 non-null    float64
 10  Region                          500 non-null    object 
dtypes: float64(8), int64(1), object(2)
memory usage: 43.1+ KB


,Year,Average_Monthly_Income,Cost_of_Living,Housing_Cost_Percentage,Tax_Rate,Savings_Percentage,Healthcare_Cost_Percentage,Education_Cost_Percentage,Transportation_Cost_Percentage
count,500.000000,500.000000,500.00000,500.000000,500.000000,500.00000,500.000000,500.000000,500.000000
mean,2011.514000,4291.248240,3716.23212,34.973700,22.400900,14.92708,12.381820,8.408440,12.475940
std,7.018284,2179.217333,1922.16053,8.657032,10.025412,8.81321,4.269823,3.775455,4.219147
min,2000.000000,534.740000,432.60000,20.100000,5.000000,0.00000,5.010000,2.000000,5.060000
25%,2005.000000,2322.360000,1967.79000,27.235000,14.742500,7.52500,8.487500,5.285000,8.752500
50%,2012.000000,4391.585000,3803.20500,35.170000,22.275000,14.95000,12.365000,8.245000,12.665000
75%,2018.000000,6233.757500,5265.96750,42.567500,30.905000,22.42750,16.222500,11.595000,15.952500
max,2023.000000,7984.510000,6996.13000,49.950000,39.920000,29.84000,19.990000,14.940000,19.970000


In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check for missing values
missing_values = data.isnull().sum()
print("Missing values in each column:\n", missing_values)

# Select only numeric columns for correlation
numeric_data = data.select_dtypes(include=["float64", "int64"])

Missing values in each column:
 Country                           0
Year                              0
Average_Monthly_Income            0
Cost_of_Living                    0
Housing_Cost_Percentage           0
Tax_Rate                          0
Savings_Percentage                0
Healthcare_Cost_Percentage        0
Education_Cost_Percentage         0
Transportation_Cost_Percentage    0
Region                            0
dtype: int64


In [17]:
# Derived metrics (PERCENT versions)
# Take-home pay (amount, after tax)
data["TakeHome"] = data["Average_Monthly_Income"] * (1 - data["Tax_Rate"] / 100.0)

# Housing share of take-home income (in PERCENT of net income)
# Example: 45 means 45% of after-tax income goes to housing
data["Housing_Share_TakeHome_Percent"] = np.where(
    net_income_share != 0,
    data["Housing_Cost_Percentage"] / net_income_share * 100,
    np.nan
)

# Savings share of take-home income (in PERCENT of net income)
data["Savings_Share_TakeHome_Percent"] = np.where(
    net_income_share != 0,
    data["Savings_Percentage"] / net_income_share * 100,
    np.nan
)

# Net savings room (difference in percentage points)
# = Savings share of net income – Housing share of net income
data["Net_Savings_Room_Percent"] = (
    data["Savings_Share_TakeHome_Percent"] - data["Housing_Share_TakeHome_Percent"]
)

# Other essentials percentage (already in % of gross income)
data["Other_Essentials_Percentage"] = (
    100
    - data["Tax_Rate"]
    - data["Housing_Cost_Percentage"]
    - data["Savings_Percentage"]
    - data["Healthcare_Cost_Percentage"]
    - data["Education_Cost_Percentage"]
    - data["Transportation_Cost_Percentage"]
)

# Savings-to-spending ratio (GROSS)
# For every 1$ you spend from gross income, how many $ you save
data["Savings_to_Spending_Ratio"] = np.where(
    spend_share != 0,
    data["Savings_Percentage"] / spend_share,   # both in %, so ratio is dimensionless
    np.nan
)

# Savings-to-spending ratio (NET, after tax)
# Use the *percent of net* as a proportion inside the formula
s_share_net = data["Savings_Share_TakeHome_Percent"] / 100.0

data["Savings_to_Spending_Ratio_Net"] = np.where(
    (1 - s_share_net) != 0,
    s_share_net / (1 - s_share_net),
    np.nan
)

# Select and order columns for export (only percentage-based "share" metrics)
cols_to_export = [
    "Country", "Year", "Region",
    "Average_Monthly_Income", "Cost_of_Living",
    "Tax_Rate", "Housing_Cost_Percentage", "Savings_Percentage",
    "Healthcare_Cost_Percentage", "Education_Cost_Percentage",
    "Transportation_Cost_Percentage",
    "TakeHome",
    "Housing_Share_TakeHome_Percent",
    "Savings_Share_TakeHome_Percent",
    "Net_Savings_Room_Percent",
    "Other_Essentials_Percentage",
    "Savings_to_Spending_Ratio",
    "Savings_to_Spending_Ratio_Net"
]

export_df = data[cols_to_export]

# Save to new CSV for Tableau
output_path = r"D:\tableau\Assignment\Group Assignment\cost_of_living_processed.csv"
export_df.to_csv(output_path, index=False)

print("New CSV saved to:", output_path)
print("New shape:", export_df.shape)



New CSV saved to: D:\tableau\Assignment\Group Assignment\cost_of_living_processed.csv
New shape: (500, 18)


In [18]:
export_df

,Country,Year,Region,Average_Monthly_Income,Cost_of_Living,Tax_Rate,Housing_Cost_Percentage,Savings_Percentage,Healthcare_Cost_Percentage,Education_Cost_Percentage,Transportation_Cost_Percentage,TakeHome,Housing_Share_TakeHome_Percent,Savings_Share_TakeHome_Percent,Net_Savings_Room_Percent,Other_Essentials_Percentage,Savings_to_Spending_Ratio,Savings_to_Spending_Ratio_Net
0,Australia,2013,Oceania,3483.92,1106.07,27.50,32.09,1.74,18.23,6.94,17.19,2525.842000,44.262069,2.400000,-41.862069,-3.69,0.017708,0.024590
1,India,2019,Asia,7771.03,5422.78,29.30,25.21,3.35,17.21,12.42,9.90,5494.118210,35.657709,4.738331,-30.919378,2.61,0.034661,0.049740
2,Russia,2004,Europe,6991.30,3972.36,22.94,40.85,15.48,15.28,7.10,9.59,5387.495780,53.010641,20.088243,-32.922398,-11.24,0.183152,0.251380
3,South Africa,2011,Africa,6628.04,6755.75,15.69,30.38,8.03,6.66,11.76,11.04,5588.100524,36.033685,9.524374,-26.509311,16.44,0.087311,0.105270
4,Brazil,2015,South America,2434.27,2656.36,12.44,49.27,25.06,9.34,3.63,15.09,2131.446812,56.269986,28.620375,-27.649612,-14.83,0.334401,0.400960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Canada,2007,North America,5238.60,6129.17,29.96,25.25,4.78,15.87,7.77,15.88,3669.115440,36.050828,6.824672,-29.226156,0.49,0.050200,0.073245
496,France,2004,Europe,2448.53,5719.11,11.97,45.67,4.81,6.67,8.91,6.69,2155.440959,51.880041,5.464046,-46.415995,15.28,0.050531,0.057799
497,Mexico,2003,North America,5255.04,5614.20,36.16,26.82,2.11,14.12,13.36,11.12,3354.817536,42.011278,3.305138,-38.706140,-3.69,0.021555,0.034181
498,Brazil,2023,South America,4549.89,2410.88,15.06,45.11,5.57,9.22,14.94,18.30,3864.676566,53.108076,6.557570,-46.550506,-8.20,0.058985,0.070178
